# Does a noise-trained NLE equal a noiseless NLE convolved with the errors?

## Why this test

The production NLE is trained with measurement noise injected: every training
star gets a random $\sigma_v$, Gaussian noise of that width is added to its
$v_{\rm los}$, and $\sigma_v$ is appended to the flow's context. The model
therefore learns

$$p_{\rm noisy}(v_{\rm obs}\mid \log R,\, \theta,\, \log r_{1/2},\, \sigma_v).$$

A second model is trained on the same simulations with no noise and no
$\sigma_v$ in its context:

$$p_{\rm clean}(v \mid \log R,\, \theta,\, \log r_{1/2}).$$

If both flows are faithful, the first is the second convolved with the error
kernel,

$$p_{\rm noisy}(v_{\rm obs}\mid\sigma_v) \;=\; \int p_{\rm clean}(v)\;
\mathcal N(v_{\rm obs};\, v,\, \sigma_v^2)\, {\rm d}v ,$$

for every $\sigma_v$ and every context. Two reasons to check it:

1. It is the paper's demonstration that the noise augmentation is learned
   consistently with the physics (the model does not use $\sigma_v$ as a
   shortcut).
2. Real errors need not be Gaussian. The noiseless model is the one that lets
   an arbitrary per-star kernel be applied at inference; the convolution
   below is that machinery.

**Models used.** Both trained on shards 0–1 (190k galaxies), T2 architecture
(NSF 4×[128,128], 8 bins, context MLP → 64), 30k steps at batch 16384:
`errtest_clean` = run `voj63drr`, `errtest_noisy` = run `6xaw28uw`
(`configs/nle_8p_v4_errtest.py`).

In [ ]:
# --- paths and imports -------------------------------------------------
import os, sys, math, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))                           # nle_dsph package
sys.path.insert(0, os.path.expanduser('~/my_modules'))  # dsph_analysis (catalogue loader)
os.chdir(ROOT)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

MODELS = Path(os.path.expanduser('~/scratch/projects/sbi_dsph/trained_models/nle/nle_8p_ZhaoPlumCOM_v4'))
OUT = Path(os.path.expanduser('~/scratch/projects/nle_dsph'))
LABELS = ('dm_alpha', 'dm_beta', 'dm_gamma', 'dm_log_rdm', 'dm_log_rho0', 'df_beta0', 'df_log_ra')

from nle import NLE                     # the Lightning module: flow + normalisation
from transforms import StarBatch        # the per-star container the model consumes
from diagnostics.pit_by_regime import best_checkpoint, load_shard

In [ ]:
clean = NLE.load_from_checkpoint(best_checkpoint(MODELS / 'voj63drr'), map_location=dev, pre_transforms=None).eval()
noisy = NLE.load_from_checkpoint(best_checkpoint(MODELS / '6xaw28uw'), map_location=dev, pre_transforms=None).eval()

# The context lists show the one structural difference: `vel_error` is in
# the noisy model's context and absent from the clean one's.
print('clean  target', clean.target_fields, ' context', clean.context_fields)
print('noisy  target', noisy.target_fields, ' context', noisy.context_fields)

## 1. Units: the flow's density is per *normalised* velocity

`NLE.log_prob` returns $\log p$ per unit of the normalised velocity
$\tilde v = (v - v_{\rm loc}) / v_{\rm scale}$, and each model has its own
`vel_scale` (the half-range of the training velocities after noise). To
compare the two models, or to feed either into a sampler alongside anything
in km/s, subtract $\log v_{\rm scale}$:

$$\log p(v)\,[{\rm per\ km/s}] = \log p(\tilde v) - \log v_{\rm scale}.$$

Forgetting this gives a constant offset of $\log(v_{\rm scale,noisy} /
v_{\rm scale,clean})$ nats per star that looks like a model disagreement
(it was the first thing this test "found").

In [ ]:
print('vel_scale  clean %.2f  noisy %.2f km/s  ->  log ratio %.4f nats/star' % (
    clean.vel_scale[0], noisy.vel_scale[0], math.log(float(noisy.vel_scale[0]) / float(clean.vel_scale[0]))))


def logp_kms(model, R, v, err, theta, lrs):
    """Per-star log p in units of 1/(km/s).

    R      (n,)   projected radius [kpc]; only |pos| enters with condition_on_radius
    v      (n,)   velocity to evaluate at [km/s]
    err    (n,)   sigma_v [km/s], or None for the clean model
    theta  (n,7)  labels in the training parametrisation (radii in units of r_half)
    lrs    float  log10 r_half [kpc]
    """
    n = len(R)
    pos = torch.zeros((n, 2), device=dev)
    pos[:, 0] = R
    b = StarBatch(pos=pos, vel=v.view(n, 1),
                  vel_error=None if err is None else err.view(n, 1),
                  theta=theta, cond=torch.full((n, 1), lrs, device=dev))
    with torch.no_grad():
        return model.log_prob(b) - torch.log(model.vel_scale[0])

## 2. One star, by eye

Take one held-out galaxy (shard 40 was never used in training), one of its
stars, and a chosen $\sigma_v$. Plot $p_{\rm clean}(v)$, its numerical
convolution with $\mathcal N(0, \sigma_v^2)$ on a fine grid, and
$p_{\rm noisy}(v_{\rm obs} \mid \sigma_v)$ evaluated directly. The
dashed curve should sit on the shaded one.

In [ ]:
pos, vel, th, lrs_all, n = load_shard(40, n_gal=50)   # (pos, vel) per star; (th, lrs, n) per galaxy

g = 3                                       # which galaxy
sl = slice(int(n[:g].sum()), int(n[:g + 1].sum()))
R = pos[sl, :2].norm(dim=1).to(dev)         # projection along z: sky plane = (x, y)
v_true = vel[sl, 2].to(dev)                 # true line-of-sight velocity
theta = th[g].expand(len(R), 7).to(dev)     # every star of a galaxy shares its theta
lrs = float(lrs_all[g])
sig_true = float(v_true.std())

i, sigma = 0, 2.0                           # star index, assumed error [km/s]
vg = torch.linspace(-30, 30, 1201, device=dev)
same = lambda k: (R[i].expand(k), theta[:k], lrs)

p_clean = torch.exp(logp_kms(clean, R[i].expand(1201), vg, None, theta[:1].expand(1201, 7), lrs)).cpu()
p_noisy = torch.exp(logp_kms(noisy, R[i].expand(1201), vg, torch.full((1201,), sigma, device=dev),
                             theta[:1].expand(1201, 7), lrs)).cpu()

# direct convolution on the grid: p_conv(v_obs) = sum_v p_clean(v) N(v_obs; v, sigma) dv
vgc = vg.cpu(); dv = float(vgc[1] - vgc[0])
kern = torch.exp(-0.5 * ((vgc[:, None] - vgc[None, :]) / sigma) ** 2) / (sigma * math.sqrt(2 * math.pi))
p_conv = (kern * p_clean[None, :]).sum(1) * dv

plt.figure(figsize=(7, 4))
plt.plot(vgc, p_clean, color='0.6', label='$p_{\\rm clean}(v)$, no noise')
plt.fill_between(vgc, p_conv, alpha=.35, label=f'$p_{{\\rm clean}} * \\mathcal{{N}}(0,{sigma}^2)$')
plt.plot(vgc, p_noisy, 'k--', label=f'$p_{{\\rm noisy}}(v_{{\\rm obs}}\\mid\\sigma_v={sigma})$')
plt.xlabel('v [km/s]'); plt.ylabel('density [1/(km/s)]'); plt.legend()
plt.title(f'galaxy {g}: $\\sigma_{{\\rm true}}$ = {sig_true:.1f} km/s;  star {i} at R = {float(R[i]):.3f} kpc')
plt.show()

## 3. The convolution for many stars at once

For a quantitative comparison we need the convolved density at every star's
observed velocity. The integrand $p_{\rm clean}(v)\,\mathcal N(v_{\rm obs};
v, \sigma_v)$ is only non-negligible where *both* factors are:

* the intrinsic peak lives in $|v| \lesssim 12\,\sigma_{\rm true}$,
* the kernel in $v_{\rm obs} \pm 8\sigma_v$.

The grid spans the **intersection** of the two, so it resolves whichever is
narrower: a 0.1 km/s galaxy under 10 km/s errors, or a 20 km/s galaxy with
0.3 km/s errors. (Gridding the union instead left 0.4 nats/star of quadrature
error in the first case.) The sum is a trapezoid rule done in log space with
`logsumexp` so very small densities do not underflow.

In [ ]:
def convolved_logp(R, v_obs, sigma, sig_true, theta, lrs, n_grid=257):
    """log Int p_clean(v) N(v_obs; v, sigma) dv, per star, in 1/(km/s).

    v_obs, sig_true (n,) tensors; sigma a float; theta (n,7); lrs float.
    n_grid trapezoid nodes per star (accuracy/cost trade-off, section 5).
    """
    lo = torch.maximum(v_obs - 8 * sigma, -12 * sig_true)
    hi = torch.minimum(v_obs + 8 * sigma, 12 * sig_true)
    hi = torch.maximum(hi, lo + 1e-3)                       # guard against an empty overlap
    t = torch.linspace(0, 1, n_grid, device=dev)
    grid = lo[:, None] + (hi - lo)[:, None] * t[None, :]    # (n, n_grid)
    B = len(v_obs)
    # one batched forward pass over all (star, node) pairs
    lp = logp_kms(clean, R.repeat_interleave(n_grid), grid.reshape(-1), None,
                  theta.repeat_interleave(n_grid, 0), lrs).view(B, n_grid)
    lk = -0.5 * ((v_obs[:, None] - grid) / sigma) ** 2 - math.log(sigma * math.sqrt(2 * math.pi))
    w = torch.zeros_like(lp); w[:, [0, -1]] = math.log(0.5)  # trapezoid end weights
    return torch.logsumexp(lp + lk + w, dim=1) + torch.log((hi - lo) / (n_grid - 1))


# d = log p_noisy - log(convolved p_clean), per star, at this galaxy's true theta
st = torch.full_like(v_true, sig_true)
for sigma in (0.3, 1.0, 3.0, 10.0):
    torch.manual_seed(0)
    v_obs = v_true + sigma * torch.randn_like(v_true)
    d = (logp_kms(noisy, R, v_obs, torch.full_like(v_obs, sigma), theta, lrs)
         - convolved_logp(R, v_obs, sigma, st, theta, lrs))
    print(f'sigma_v = {sigma:5.1f} km/s:  per star  mean {d.mean():+.4f}  rms {d.pow(2).mean().sqrt():.4f}'
          f'   |  summed over the galaxy ({len(d)} stars) {d.sum():+.2f} nats')

**Reading it.** The per-star mean of $d$ is at the $10^{-3}$ level and the
galaxy sum below a nat. The rms per star (a few $\times 10^{-2}$) is the
*combined* approximation error of two independently trained flows plus the
quadrature; it adds up incoherently over a galaxy ($\sqrt{N}$), which is why
the galaxy sum is small. For the posterior what matters is the galaxy-level
error, since every star is evaluated at the same $\theta$.

## 4. The batch result (3000 galaxies, four $\sigma_v$)

`diagnostics/check_error_convolution.py` runs the same comparison on 3000
galaxies of shard 40 and also compares against the 5-shard T2 model
(`noisy2`). Columns: `sigma` the assumed error (km/s); `sig_lo`–`sig_hi` the
bin of the galaxy's intrinsic dispersion (km/s); `star_*` statistics of $d$
per star; `gal_*` of $d$ summed over a galaxy; `nll_*` the mean negative log
density of each side. The figure scans $\log L$ in `dm_log_rho0` for six
galaxies with the other labels at the truth: the three curves should
coincide.

In [ ]:
csv = sorted((MODELS / 'voj63drr').glob('error_convolution_*v2.csv'))
if csv:
    display(pd.read_csv(csv[-1]).round(4))
    display(Image(str(csv[-1]).replace('.csv', '.png')))
else:
    print('batch run not found; run  python diagnostics/check_error_convolution.py --clean', MODELS / 'voj63drr', '--noisy', MODELS / '6xaw28uw')

## 5. What the noiseless route costs

Each convolved evaluation is `n_grid` forward passes of the clean model, so
its cost relative to one noisy-model evaluation is about `1.1 × n_grid`. The
accuracy floor worth reaching is the model-to-model scatter above (~1 nat per
galaxy): 65 nodes get there, 129 are well below it. The kernel shape does not
change this; a heavy-tailed kernel only widens the grid.

In [ ]:
sigma = 1.0
torch.manual_seed(0); v_obs = v_true + sigma * torch.randn_like(v_true); e = torch.full_like(v_obs, sigma)
ref = convolved_logp(R, v_obs, sigma, st, theta, lrs, n_grid=1025)


def timeit(f, reps=5):
    f(); torch.cuda.synchronize(); t0 = time.time()
    for _ in range(reps): f()
    torch.cuda.synchronize(); return (time.time() - t0) / reps


t_noisy = timeit(lambda: logp_kms(noisy, R, v_obs, e, theta, lrs))
print(f'{"n_grid":>7} {"time / noisy":>13} {"rms error vs 1025 nodes [nats/star]":>38}')
for ng in (17, 33, 65, 129, 257):
    t = timeit(lambda: convolved_logp(R, v_obs, sigma, st, theta, lrs, n_grid=ng), reps=2)
    err_ng = (convolved_logp(R, v_obs, sigma, st, theta, lrs, n_grid=ng) - ref).pow(2).mean().sqrt()
    print(f'{ng:>7} {t / t_noisy:>12.1f}x {float(err_ng):>38.4f}')